# 06 — Cold-Start Handling

**Objective**: show how `AryInfFallBack` (`src/models/arycolbring/inout/fallback_reasoner.py`) cleans up a model's Top-N recommendations by removing already-purchased items and backfilling the gaps -- the standard cold-start / repeat-purchase problem every recommender faces in production.

**Audience**: anyone wiring `AryColBringInference.recommend()` output into a real product surface (an already-purchased item showing up as a "recommendation" is a bad user experience, and a naive top-N list runs short once you filter those out).

`AryInfFallBack` is pure NumPy/pandas (no Cython extension involved), so -- unlike the training/inference notebooks -- **this one runs directly against the real class**, no mock needed.

## 1. The problem

A model's raw Top-N candidate pool often includes items the user already bought. `AryInfFallBack.clean_recommendations()` runs a 3-stage pipeline:

1. **Filter**: drop candidates already in the user's purchase history.
2. **Patch**: if the model's overscanned candidate pool has enough leftover items, use those to fill the gap.
3. **Item-to-item fallback**: if the model pool is *still* short, fall back to cosine similarity over `item_embeddings`, seeded by the user's purchase history -- this is the actual cold-start path.

## 2. Setup: item embeddings + purchase history

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from src.models.arycolbring.inout.fallback_reasoner import AryInfFallBack

rng = np.random.default_rng(0)
N_ITEMS, DIM = 30, 8
item_embeddings = rng.normal(size=(N_ITEMS, DIM))

# Purchase history: AryInfFallBack auto-detects the user/item columns
# regardless of naming (via DetectReco_Identifier), so this works even
# if your raw data calls them something else.
purchase_data = pd.DataFrame({
    "user_id": [1, 1, 1, 2, 2],
    "item_id": [0, 1, 2, 5, 6],
})

fallback = AryInfFallBack(purchase_data=purchase_data, item_embeddings=item_embeddings)
print("Resolved columns:", fallback.user_col, fallback.item_col)
print("User 1 has already purchased:", fallback.purchased_items(1))

## 3. A shallow candidate pool (typical cold-start trigger)

Suppose the model's Top-5 candidates for user 1 include 2 items they've already bought -- a shallow, non-overscanned pool like this is exactly what pushes `clean_recommendations()` into the item-to-item fallback path.

In [ ]:
candidate_pool = [(0, 0.90), (1, 0.85), (7, 0.80), (12, 0.75), (3, 0.70)]
#                  ^already bought  ^already bought

result = fallback.clean_recommendations(user_id=1, candidate_pool=candidate_pool, n_items=5)
result

## 4. Checking the result

In [ ]:
bought = fallback.purchased_items(1)
assert not (set(result["item_id"]) & bought), "no already-purchased item should appear"
assert len(result) == 5, "the gap left by filtering must be fully backfilled"

n_from_model    = (result["source"] == fallback.SOURCE_MODEL).sum()
n_from_fallback = (result["source"] == fallback.SOURCE_ITEM2ITEM).sum()
print(f"{n_from_model} slot(s) filled directly from the model's pool")
print(f"{n_from_fallback} slot(s) backfilled via item-to-item cosine similarity")
print("\nAll checks passed -- no repurchase, list still fully populated.")

## 5. A brand-new user (true cold start)

`AryInfFallBack` doesn't require the *model* to know about a user -- item-to-item fallback only needs **some** seed items (e.g. items viewed in the current session, or a "similar users" heuristic upstream) to work. If a user has zero purchase history, `item_to_item_candidates()` can still be called directly with a manual seed list:

In [ ]:
# A brand-new user with no purchase history -- seed the fallback with
# whatever signal you do have (e.g. items viewed this session).
session_viewed = [4, 9]
candidates = fallback.item_to_item_candidates(seed_items=session_viewed, exclude=set(session_viewed), n=5)
for item_id, sim in candidates:
    print(f"item {item_id:>3}  cosine similarity to session = {sim:.3f}")

## Summary

- `AryInfFallBack(purchase_data, item_embeddings)` wraps a trained model's raw recommendations with purchase-aware filtering + backfilling.
- `clean_recommendations()` -- the normal per-user path -- always returns a full-length list even after filtering out repurchases.
- `item_to_item_candidates()` -- the true cold-start path -- works from any seed item list, purchase history or otherwise, with no dependency on the model having seen the user before.